# Week 88, Day 4: Tests

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w088d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('waymark_domain.py', 'w', encoding='utf-8') as f:
    f.write('import heapq\n\n\nclass UnknownPlace(LookupError):\n    pass\n\n\nclass NoRoute(Exception):\n    pass\n\n\ndef naismith_minutes(km, climb_m):\n    return round(km / 5 * 60 + climb_m / 600 * 60)\n\n\nclass Route:\n    def __init__(self, stops, km, climb_m):\n        self.stops, self.km, self.climb_m = list(stops), km, climb_m\n\n    @property\n    def minutes(self):\n        return naismith_minutes(self.km, self.climb_m)\n\n    def __repr__(self):\n        return f"Route({self.stops}, {self.km} km, {self.climb_m} m)"\n\n\nclass PathNetwork:\n    def __init__(self, heights):\n        self.heights = dict(heights)\n        self.paths = {name: {} for name in self.heights}\n\n    def _check(self, *names):\n        for name in names:\n            if name not in self.heights:\n                raise UnknownPlace(name)\n\n    def add_path(self, a, b, km):\n        self._check(a, b)\n        if km <= 0:\n            raise ValueError("km must be positive")\n        self.paths[a][b] = km\n        self.paths[b][a] = km\n\n    def climb(self, stops):\n        return sum(max(0, self.heights[b] - self.heights[a]) for a, b in zip(stops, stops[1:]))\n\n    def shortest(self, start, end):\n        self._check(start, end)\n        dist, prev, done = {start: 0}, {}, set()\n        heap = [(0, start)]\n        while heap:\n            d, node = heapq.heappop(heap)\n            if node in done:\n                continue\n            done.add(node)\n            if node == end:\n                break\n            for nxt, km in sorted(self.paths[node].items()):\n                if d + km < dist.get(nxt, float("inf")):\n                    dist[nxt] = d + km\n                    prev[nxt] = node\n                    heapq.heappush(heap, (d + km, nxt))\n        if end not in done:\n            raise NoRoute(f"no route from {start} to {end}")\n        stops = [end]\n        while stops[-1] != start:\n            stops.append(prev[stops[-1]])\n        stops.reverse()\n        return Route(stops, round(dist[end], 2), self.climb(stops))\n')
with open('pace.py', 'w', encoding='utf-8') as f:
    f.write('import statistics\nfrom waymark_domain import naismith_minutes\n\n\ndef pace_factor(walks, minimum=3):\n    if len(walks) < minimum:\n        return 1.0\n    return round(statistics.median(w["minutes"] / naismith_minutes(w["km"], w["climb_m"]) for w in walks), 2)\n\n\ndef personal_minutes(km, climb_m, walks):\n    return round(naismith_minutes(km, climb_m) * pace_factor(walks))\n')
with open('map_check.py', 'w', encoding='utf-8') as f:
    f.write('import csv\nimport io\nfrom collections import deque\n\ndef read_junctions(text):\n    reader = csv.DictReader(io.StringIO(text))\n    if reader.fieldnames != ["name", "height_m"]:\n        return {}, ["line 1: expected columns name,height_m"]\n    heights, problems = {}, []\n    for line, row in enumerate(reader, start=2):\n        name = (row["name"] or "").strip()\n        try:\n            height = int((row["height_m"] or "").strip())\n        except ValueError:\n            problems.append(f"line {line}: height must be a whole number")\n            continue\n        if not name:\n            problems.append(f"line {line}: missing name")\n        elif name in heights:\n            problems.append(f"line {line}: duplicate junction {name}")\n        else:\n            heights[name] = height\n    return heights, problems\n\ndef read_paths(text, heights):\n    reader = csv.DictReader(io.StringIO(text))\n    if reader.fieldnames != ["from", "to", "km"]:\n        return [], ["line 1: expected columns from,to,km"]\n    paths, problems = [], []\n    for line, row in enumerate(reader, start=2):\n        a, b = (row["from"] or "").strip(), (row["to"] or "").strip()\n        unknown = next((n for n in (a, b) if n not in heights), None)\n        if not a or not b:\n            problems.append(f"line {line}: missing junction")\n            continue\n        try:\n            km = float(row["km"])\n        except (TypeError, ValueError):\n            km = 0\n        if unknown:\n            problems.append(f"line {line}: unknown junction {unknown}")\n        elif a == b:\n            problems.append(f"line {line}: path from {a} to itself")\n        elif km <= 0:\n            problems.append(f"line {line}: km must be a positive number")\n        else:\n            paths.append((a, b, km))\n    return paths, problems\n\ndef outside_largest(names, paths):\n    neighbours = {n: set() for n in names}\n    for a, b, _ in paths:\n        neighbours[a].add(b)\n        neighbours[b].add(a)\n    seen, parts = set(), []\n    for start in sorted(neighbours):\n        if start not in seen:\n            seen.add(start)\n            queue, part = deque([start]), []\n            while queue:\n                node = queue.popleft()\n                part.append(node)\n                for nxt in neighbours[node] - seen:\n                    seen.add(nxt)\n                    queue.append(nxt)\n            parts.append(sorted(part))\n    parts.sort(key=lambda p: (-len(p), p[0]))\n    return sorted(n for p in parts[1:] for n in p)\n\ndef check_map(junctions_text, paths_text, steep_limit=250):\n    heights, junction_errors = read_junctions(junctions_text)\n    paths, path_errors = read_paths(paths_text, heights)\n    warnings = []\n    steep = []\n    for a, b, km in paths:\n        rate = abs(heights[b] - heights[a]) / km\n        if rate > steep_limit:\n            steep.append((a, b, round(rate)))\n    warnings += [f"steep path {a} to {b} ({rate} m per km)" for a, b, rate in sorted(steep, key=lambda s: -s[2])]\n    counts = {}\n    for a, b, _ in paths:\n        key = tuple(sorted((a, b)))\n        counts[key] = counts.get(key, 0) + 1\n    warnings += [f"duplicate path between {a} and {b}" for (a, b), n in sorted(counts.items()) if n > 1]\n    apart = outside_largest(heights, paths)\n    if apart:\n        warnings.append("not connected to the rest: " + ", ".join(apart))\n    errors = junction_errors + path_errors\n    return {"heights": heights, "paths": paths, "errors": errors, "warnings": warnings, "ok": not errors,\n            "summary": f"{len(heights)} junctions, {len(paths)} paths, {len(errors)} errors, {len(warnings)} warnings"}\n')

**Predict before you run:** what will this print?



In [ ]:
import random
from waymark_domain import PathNetwork

rng = random.Random(3)
names = [f"J{i}" for i in range(5)]
network = PathNetwork({n: rng.randint(0, 300) for n in names})
for i in range(1, 5):
    network.add_path(names[i], names[rng.randrange(i)], round(rng.uniform(0.5, 3), 1))
print(sorted(network.heights.items()))
print({a: sorted(b) for a, b in network.paths.items()})
again = random.Random(3)
print([again.randint(0, 300) for _ in range(5)] == [network.heights[n] for n in names])

In [ ]:
import itertools
import random
from waymark_domain import PathNetwork

def all_simple_routes(network, start, end, path=None):
    path = path or [start]
    if path[-1] == end:
        yield path
        return
    for nxt in network.paths[path[-1]]:
        if nxt not in path:
            yield from all_simple_routes(network, start, end, path + [nxt])

network = PathNetwork({"A": 0, "B": 0, "C": 0, "D": 0})
for a, b, km in [("A", "B", 2), ("B", "C", 3), ("A", "C", 6), ("C", "D", 1), ("B", "D", 5)]:
    network.add_path(a, b, km)
routes = list(all_simple_routes(network, "A", "D"))
lengths = {tuple(r): sum(network.paths[x][y] for x, y in zip(r, r[1:])) for r in routes}
print(len(routes), min(lengths.values()), network.shortest("A", "D").km)

> **Key idea.** Find gaps from evidence: stories without tests, unhappy paths never triggered, hard-to-test parts, and mutations no test would catch. Use builders with defaults so each test states only what matters. Check algorithms against brute-force oracles on many seeded random inputs, and test properties that need no oracle. Compare long outputs with golden files and show diffs; regenerate deliberately. Fuzz parsers with damaged input and check they never crash and their reports stay consistent. Keep tests independent of each other, the clock and chance.

### Exercise: A walk builder

Write `WalkBuilder` with defaults `day="2026-01-01"`, `km=5.0`, `climb_m=0`, `minutes=60`, `route=None`, and chainable methods `on(day)`, `km(value)`, `climb(metres)`, `taking(minutes)` and `along(route)`, each returning the builder. `build()` returns a **new** dict each time with the keys `day`, `km`, `climb_m`, `minutes` and `route`. Then `walks(n, start="2026-01-01", every_days=7, **changes)`: a list of `n` built walks on days `every_days` apart from `start`, each with the defaults overridden by `changes` (keyword names as the dict keys, such as `km=8.0`).

In [ ]:
from datetime import date, timedelta

class WalkBuilder:
    ___

def walks(n, start="2026-01-01", every_days=7, **changes):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_builder():\n    \"one readable line per walk\"\n    b = WalkBuilder().on(\"2026-03-29\").km(15.5).climb(135)\n    w1, w2 = b.build(), b.build()\n    assert w1 == {\"day\": \"2026-03-29\", \"km\": 15.5, \"climb_m\": 135, \"minutes\": 60, \"route\": None} and w1 is not w2\n    w1[\"km\"] = 0\n    assert b.build()[\"km\"] == 15.5\n    assert WalkBuilder().taking(95).along(\"Ridge walk\").build()[\"route\"] == \"Ridge walk\"\n    many = walks(3, km=8.0, minutes=100)\n    assert [w[\"day\"] for w in many] == [\"2026-01-01\", \"2026-01-08\", \"2026-01-15\"] and all(w[\"km\"] == 8.0 and w[\"minutes\"] == 100 for w in many)\n    assert walks(2, start=\"2026-12-30\", every_days=3)[1][\"day\"] == \"2027-01-02\"")

### Exercise: An oracle for route search

Using the page's `waymark_domain`, write `random_network(rng, n, extra)`: junctions `J0` to `J{n-1}` with heights `rng.randint(0, 300)` (drawn in order), then for `i` from 1 to `n − 1` a path from `J{i}` to `J{rng.randrange(i)}` of `round(rng.uniform(0.5, 3), 1)` km, then `extra` more paths between `rng.sample(names, 2)` with the same kind of length, skipping any pair already joined. Then `brute_force_km(network, start, end)`: the shortest distance over **all** simple routes, rounded to 2 places. And `disagreement(rng, trials, n=6, extra=4)`: for each trial build a random network and compare `shortest(...).km` with `brute_force_km` for every ordered pair of different junctions; return the first `(network, start, end)` where they differ, or `None`.

In [ ]:
import random
from waymark_domain import PathNetwork

def random_network(rng, n, extra):
    ___

def brute_force_km(network, start, end):
    ___

def disagreement(rng, trials, n=6, extra=4):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nimport heapq\n\ndef test_generator():\n    \"seeded and connected\"\n    a = random_network(random.Random(3), 6, 4)\n    b = random_network(random.Random(3), 6, 4)\n    assert a.heights == b.heights and a.paths == b.paths and len(a.heights) == 6\n    rng = random.Random(3)\n    assert [rng.randint(0, 300) for _ in range(6)] == [a.heights[f\"J{i}\"] for i in range(6)], \"heights first, in order\"\n    edges = sum(len(v) for v in a.paths.values()) // 2\n    assert 5 <= edges <= 9\n\ndef test_oracle():\n    \"agreement on the real code, a disagreement on a broken one\"\n    assert disagreement(random.Random(1), 30) is None\n    import waymark_domain\n    original = waymark_domain.PathNetwork.shortest\n    def finished_too_soon(self, start, end):\n        dist, prev, done, heap = {start: 0}, {}, {start}, [(0, start)]\n        while heap:\n            d, node = heapq.heappop(heap)\n            for nxt, km in sorted(self.paths[node].items()):\n                if nxt not in done:\n                    done.add(nxt)\n                    dist[nxt], prev[nxt] = d + km, node\n                    heapq.heappush(heap, (d + km, nxt))\n        stops = [end]\n        while stops[-1] != start:\n            stops.append(prev[stops[-1]])\n        return waymark_domain.Route(stops[::-1], round(dist[end], 2), 0)\n    waymark_domain.PathNetwork.shortest = finished_too_soon\n    try:\n        found = disagreement(random.Random(1), 30)\n    finally:\n        waymark_domain.PathNetwork.shortest = original\n    assert found is not None\n    network, start, end = found\n    assert brute_force_km(network, start, end) < finished_too_soon(network, start, end).km")

### Exercise: Golden files

Write `check_golden(output, path, update=False)`: if the file doesn't exist, or `update` is true, write `output` to it and return `"written"`; if it holds exactly `output`, return `"same"`; otherwise return the differences as a unified diff (`difflib.unified_diff` of the two texts' lines with their line endings, `fromfile="golden"`, `tofile="actual"`) joined into one string.

In [ ]:
import difflib
import os

def check_golden(output, path, update=False):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import os\n\nREPORT = \"Walks: 9\\nDistance: 74.5 km\\nPace: 4% slower\\n\"\n\ndef test_golden():\n    \"write, compare, diff\"\n    if os.path.exists(\"year.golden\"):\n        os.remove(\"year.golden\")\n    assert check_golden(REPORT, \"year.golden\") == \"written\"\n    assert check_golden(REPORT, \"year.golden\") == \"same\"\n    diff = check_golden(REPORT.replace(\"74.5\", \"75.0\"), \"year.golden\")\n    assert diff.startswith(\"--- golden\\n+++ actual\\n\") and \"-Distance: 74.5 km\\n\" in diff and \"+Distance: 75.0 km\\n\" in diff\n    assert check_golden(\"new\\n\", \"year.golden\", update=True) == \"written\" and open(\"year.golden\").read() == \"new\\n\"")

### Exercise: Tests that catch pace bugs

`pace.py` (a page file) has Day 2's `pace_factor(walks, minimum=3)` and `personal_minutes(km, climb_m, walks)`. Write pytest tests, using a small builder or plain dicts, that pass on it and catch all five broken versions.

In [ ]:
import pytest
from pace import pace_factor, personal_minutes

def test_default_with_no_walks():
    assert pace_factor([]) == 1.0

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"pace.py\").read()\nBROKEN = {\n    \"mean, not median\": GOOD.replace(\"statistics.median(\", \"statistics.mean(\"),\n    \"needs more walks\": GOOD.replace(\"if len(walks) < minimum:\", \"if len(walks) <= minimum:\"),\n    \"upside down\": GOOD.replace('w[\"minutes\"] / naismith_minutes(w[\"km\"], w[\"climb_m\"])', 'naismith_minutes(w[\"km\"], w[\"climb_m\"]) / w[\"minutes\"]'),\n    \"rounded too far\": GOOD.replace(\"), 2)\", \"), 1)\"),\n    \"ignores the factor\": GOOD.replace(\"return round(naismith_minutes(km, climb_m) * pace_factor(walks))\", \"return naismith_minutes(km, climb_m)\"),\n}\ndef test_suite():\n    \"all five caught\"\n    for name, code in BROKEN.items():\n        assert code != GOOD, name\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and r[\"passed\"], r[\"output\"][-600:]\n    results = against(source, \"pace\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"pace.py\", GOOD)\n    missed = [k for k in BROKEN if not results[k]]\n    assert results[\"correct\"] == [] and not missed, f\"not caught: {missed}\"")

### Bug hunt: A check that checked nothing

The property loop compares routes for every pair `(a, b)` with `a < b`, but it builds the pairs from an exhausted iterator, so it compares nothing and reports success: it prints `compared 0 pairs`. Make the names a list, so it prints `compared 6 pairs`.

In [ ]:
names = iter(["A", "B", "C", "D"])
pairs = [(a, b) for a in names for b in names if a < b]
compared = 0
for a, b in pairs:
    compared += 1
print("compared", compared, "pairs")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"count what you check\"\n    expect_output(\"compared 6 pairs\")")

### Bug hunt: Tests that share a list

Both tests use one module-level list, so the second test sees the first test's walk and fails: it prints `test_one ok` then `test_two FAILED`. Give each test its own list (build it inside the test), so both pass.

In [ ]:
LOG = []

def test_one():
    LOG.append({"km": 5.0})
    assert len(LOG) == 1

def test_two():
    LOG.append({"km": 8.0})
    assert len(LOG) == 1

for test in (test_one, test_two):
    try:
        test()
        print(test.__name__, "ok")
    except AssertionError:
        print(test.__name__, "FAILED")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"independent tests\"\n    expect_output(\"test_one ok\\ntest_two ok\")")

### Puzzle: A property test

Arrange a property test: for many seeded random networks, every route's distance must equal the sum of its paths. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the number of paths in a random tree of 7 junctions
q = ___   # "oracle" or "golden file": a slow, obviously correct implementation used to check a fast one
r = ___   # True or False: a test that fails one run in fifty is still useful as it is
s = ___   # the number of simple routes from A to C in a triangle A-B, B-C, A-C
t = ___   # what WalkBuilder().km(3).km(7).build()["km"] is

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 6 and q == \"oracle\" and r is False, \"p, q and r\"\n    assert s == 2 and t == 7, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: fuzzing the map checker

Write `fuzz(check, rng, trials=300)` for a function `check(junctions_text, paths_text)` like the page's `map_check.check_map`. For each trial, build valid files: `n = rng.randint(2, 6)` junctions `J0`... with heights `rng.randint(0, 300)`, and `rng.randint(1, 6)` paths between `rng.sample(names, 2)` with `round(rng.uniform(0.1, 3), 1)` km. Then damage each **data** line of each file with probability 0.3: replace it with one of (chosen with `rng.choice`) its first field only, the line with its last field emptied, the line with its last field replaced by `x`, or the line repeated twice (as two lines). Call `check`. Return `(junctions_text, paths_text, problem)` for the first trial where the call **raises** (`problem` is `"crashed: ExceptionName"`), where `ok` isn't the same as "no errors" (`"ok is wrong"`), or where an error doesn't start with `line N: ` for a line number `N` between 2 and the number of lines in a file (`"bad line number"`). Return `None` if every trial passes. It must pass the page's checker and catch the crash in Day 3's original reader.

In [ ]:
import random
import re

def fuzz(check, rng, trials=300):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nimport map_check\n\ndef _buggy():\n    code = open(\"map_check.py\").read()\n    code = code.replace('a, b = (row[\"from\"] or \"\").strip(), (row[\"to\"] or \"\").strip()', 'a, b = row[\"from\"].strip(), row[\"to\"].strip()')\n    code = code.replace('''        if not a or not b:\n            problems.append(f\"line {line}: missing junction\")\n            continue\n''', \"\")\n    ns = {}\n    exec(code, ns)\n    return ns[\"check_map\"]\n\ndef test_fuzz():\n    \"robust checker passes; the original crashes\"\n    assert fuzz(map_check.check_map, random.Random(2)) is None\n    found = fuzz(_buggy(), random.Random(2))\n    assert found is not None and found[2] == \"crashed: AttributeError\", found\n    jt, pt, problem = found\n    assert jt.startswith(\"name,height_m\\n\") and pt.startswith(\"from,to,km\\n\")\n\ndef test_invariants():\n    \"a lying checker is caught too\"\n    def liar(jt, pt):\n        r = map_check.check_map(jt, pt)\n        return dict(r, ok=True)\n    assert fuzz(liar, random.Random(2))[2] == \"ok is wrong\"\n    def miscounts(jt, pt):\n        r = map_check.check_map(jt, pt)\n        return dict(r, errors=[e.replace(\"line \", \"line 1\", 1) for e in r[\"errors\"]])\n    assert fuzz(miscounts, random.Random(2))[2] == \"bad line number\"")

## Warm-up

Joining each new junction to one earlier one always gives a **connected** network (a random tree), and the seed makes it the same every run. Generators like this let one test check thousands of networks, which is how the bugs nobody thought of get found.

## Where the gaps are

Start the tests day with evidence, not a feeling:

- **Trace stories to tests** (Week 87's `trace`): any must without a test is the first gap.
- **Look at the unhappy paths**: every `raise` in the code and every error status in the API should have a test that makes it happen.
- **Look at what's hard to test** (dates, randomness, the database, long outputs) and make it easy: inject it, build it, or compare it with a file.
- **Ask what a bug would look like**: the mutation question from Week 72. If changing `<=` to `<` in the pace code wouldn't fail a test, there's a gap.

## Builders

Tests that build their data by hand repeat the same twelve fields, and hide the one that matters. A **builder** has a sensible default for everything and a method for each thing a test might want to change, so a test reads like its scenario: `WalkBuilder().on("2026-01-04").km(15.5).build()`. Each method returns the builder (so calls chain) and `build()` returns a fresh dict each time. When a field is added to walks, only the builder changes.

## Oracles and properties

Dijkstra's algorithm is easy to get subtly wrong (Week 87's bug hunt finished nodes too early) and hard to test by hand on more than a few graphs. But there's a slow, obviously correct way to find the shortest route on a small network: try **every** simple path and take the shortest. That's an **oracle**. A property test generates many random small networks, asks both, and fails with the first network where they disagree.

Other useful properties for Waymark need no oracle at all: a route starts and ends where it should; its distance equals the sum of its paths; reversing it gives the same distance; the climb there plus the climb back equals the sum of all the height changes. Hypothesis (Week 72) automates this and shrinks failures; the idea works with a seeded `random.Random` too.

## Golden files and fuzzing

Some outputs are long: the year report, a CLI's help text, an exported map. Asserting them line by line is tedious; instead, save a known-good copy as a **golden file**, and have the test compare the current output with it, showing a **diff** when they differ. When the output changes on purpose, regenerate the file (with a flag) and review the diff in the pull request like any other change.

**Fuzzing** turns property testing on input parsers: generate valid input, damage it at random (cut a line short, empty a field, put a word where a number goes, repeat a line), and check that the parser never crashes and that its report stays consistent. Parsers are where untrusted data enters a program, so they're where fuzzing pays most.

Whatever the technique, tests must be **independent**: no shared state between tests (each builds its own data), no dependence on the real clock (inject it) or on unseeded randomness (seed it), and no dependence on the order they run in. A test that sometimes fails is worse than none, because people learn to ignore it.

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **Find the gaps** with evidence: trace stories to tests, list every `raise` without a test, and run your mutation question on one important function.
2. **One builder** for your most-used test data, and convert three tests to use it.
3. **One property test** with a seeded generator, against an oracle if you can write one.
4. **One golden-file or fuzz test** for your longest output or most exposed parser.
5. **Merge**, and note how many tests the project has now.

## Before you go

Run your whole test suite three times in a row and in random order if you can (the `pytest-randomly` plugin does it on your own computer). Any test whose result changes is tomorrow's first job. Tomorrow is **code review**: reading the sprint's changes as a reviewer would.